In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# =============================
# Paramètres fixes
# =============================
T_total = 40 * 5  # heures par semaine
T_consult_mean = 0.25  # 15 min par consultation
simulations = 3000
demand_lambda = 20
no_show_prob = 0.1
continuity_threshold = 0.7
delegation_time_per_patient = 0.05
delegation_factor = 0.7

# =============================
# Fonction de simulation
# =============================
def simulate_income(capitation_rate, p_FFS, bonus_max, prop_chronique, scenario):
    P_mean = 1000
    patient_type_probs = [0.4, 0.4, prop_chronique]
    complexity_weights = [0.8, 1.0, 1.5]

    incomes = []
    for _ in range(simulations):
        P_sim = np.random.poisson(P_mean)
        patient_types = np.random.choice([0,1,2], size=P_sim, p=patient_type_probs)
        patient_weights = np.array([complexity_weights[t] for t in patient_types])

        delegated_time = delegation_time_per_patient * P_sim * delegation_factor if scenario == 'capitation-heavy' else 0

        daily_visits = np.random.poisson(demand_lambda, 5)
        daily_visits = np.random.binomial(daily_visits, 1 - no_show_prob)
        total_visits = daily_visits.sum()

        max_visits_time = T_total / T_consult_mean - delegated_time
        total_visits = min(total_visits, max_visits_time)

        if scenario == 'FFS-dominant':
            income_FFS = p_FFS * total_visits
            income_cap = capitation_rate * patient_weights.sum() * 0.2
        elif scenario == 'capitation-heavy':
            income_FFS = p_FFS * total_visits * 0.3
            income_cap = capitation_rate * patient_weights.sum()
        else:  # mixte
            income_FFS = p_FFS * total_visits * 0.5
            income_cap = capitation_rate * patient_weights.sum() * 0.5

        proportion_seen = min(total_visits / P_sim, 1)
        bonus = bonus_max * min(proportion_seen / continuity_threshold, 1)

        incomes.append(income_FFS + income_cap + bonus)

    return np.array(incomes)

# =============================
# Dashboard complet interactif
# =============================
def interactive_dashboard(capitation_rate=100, p_FFS=50, bonus_max=1000, prop_chronique=0.2):
    scenarios = ['FFS-dominant', 'capitation-heavy', 'mixed']
    means, stds = [], []

    plt.figure(figsize=(18,10))

    # Histogrammes et stats
    for idx, scenario in enumerate(scenarios, 1):
        incomes = simulate_income(capitation_rate, p_FFS, bonus_max, prop_chronique, scenario)
        means.append(incomes.mean())
        stds.append(incomes.std())

        plt.subplot(2,3,idx)
        plt.hist(incomes, bins=50, alpha=0.6, color='skyblue', edgecolor='black')
        plt.title(f"{scenario}\nMoyenne: ${incomes.mean():,.0f} | Écart-type: ${incomes.std():,.0f}")
        plt.xlabel("Revenu ($)")
        plt.ylabel("Fréquence")

    # Barres comparatives revenu moyen / écart-type
    x = np.arange(len(scenarios))
    width = 0.35

    plt.subplot(2,1,2)
    plt.bar(x - width/2, means, width, label='Revenu moyen ($)', color='skyblue', edgecolor='black')
    plt.bar(x + width/2, stds, width, label='Écart-type ($)', color='salmon', edgecolor='black')
    plt.xticks(x, scenarios)
    plt.ylabel('Valeur ($)')
    plt.title('Comparaison du revenu moyen et de l’écart-type par scénario')
    plt.legend()

    plt.tight_layout()
    plt.show()

# =============================
# Sliders interactifs
# =============================
interact(
    interactive_dashboard,
    capitation_rate=FloatSlider(value=100, min=50, max=200, step=5, description='Capitation $'),
    p_FFS=FloatSlider(value=50, min=20, max=100, step=5, description='FFS $'),
    bonus_max=FloatSlider(value=1000, min=0, max=3000, step=100, description='Bonus max $'),
    prop_chronique=FloatSlider(value=0.2, min=0.0, max=0.5, step=0.05, description='% chroniques')
)

interactive(children=(FloatSlider(value=100.0, description='Capitation $', max=200.0, min=50.0, step=5.0), Flo…

<function __main__.interactive_dashboard(capitation_rate=100, p_FFS=50, bonus_max=1000, prop_chronique=0.2)>